# Script for downloading documents from tweede kamer API


In [6]:
# imports
import requests
import csv
import json
from pathlib import Path
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
from tqdm import tqdm
from bs4 import BeautifulSoup
import pandas as pd

import pdfplumber
from io import BytesIO
from urllib.parse import urljoin

## part 1: making json/csv with links to pdf's

In [3]:
# Configuration
BASE_URL = "https://opendata.rijksoverheid.nl/v1/documents"
OUTPUT_JSON = "beleidsnotas_rijksoverheid.json"
OUTPUT_CSV = "beleidsnotas_rijksoverheid.csv"

DOC_TYPE = "beleidsnota"   # <-- uit <name> in /v1/documents/infotypes
ROWS = 200                  # max per call (API-max)
MAX_RECORDS = 50000         # veiligheidslimiet


In [ ]:
# HTTP sessie met retries
def make_session():
    sess = requests.Session()
    sess.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/605.1.15 (KHTML, like Gecko) "
            "Version/18.1.1 Safari/605.1.15"
        ),
        "Accept": "application/json",
        "Accept-Language": "nl-NL,nl;q=0.9,en;q=0.8",
    })
    retries = Retry(
        total=3,
        connect=3,
        read=3,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retries)
    sess.mount("https://", adapter)
    sess.mount("http://", adapter)
    return sess


# Hoofdlogica: alle vergaderstukken ophalen

def fetch_vergaderstukken():
    session = make_session()
    offset = 0
    all_docs = []

    # --- Probeer total op te halen, maar crash niet als structuur anders is ---
    total_found = None
    try:
        test_params = {
            "type": DOC_TYPE,
            "output": "json",
            "rows": 1,
            "offset": 0,
        }
        test_resp = session.get(BASE_URL, params=test_params, timeout=30)
        test_json = test_resp.json()

        # Alleen gebruiken als het een dict is en 'total' bevat
        if isinstance(test_json, dict) and "total" in test_json:
            total_found = test_json["total"]
    except Exception as e:
        print(f"[WARN] Kon 'total' niet bepalen: {e}")

    # --- Setup tqdm ---
    if total_found:
        print(f"[INFO] API geeft totaal: {total_found}")
        total_batches = (total_found // ROWS) + 2
        pbar = tqdm(total=total_batches, desc="Ophalen batches")
    else:
        print("[INFO] Geen totaal beschikbaar — gebruik open-ended tqdm")
        pbar = tqdm(desc="Ophalen batches")

    # --- Ophalen batches ---
    while True:
        params = {
            "type": DOC_TYPE,
            "output": "json",
            "rows": ROWS,
            "offset": offset,
        }

        r = session.get(BASE_URL, params=params, timeout=30)
        if r.status_code != 200:
            print(f"[WARN] HTTP {r.status_code} bij offset={offset}, stop.")
            break

        try:
            data = r.json()
        except Exception as e:
            print(f"[WARN] JSON parse error bij offset={offset}: {e}")
            break

        docs = data.get("documents", []) if isinstance(data, dict) else data
        if not docs:
            print("[INFO] Geen documenten meer (lege batch), klaar.")
            break

        all_docs.extend(docs)
        offset += ROWS
        pbar.update(1)

        if offset >= MAX_RECORDS:
            print(f"[INFO] MAX_RECORDS ({MAX_RECORDS}) bereikt, stoppen.")
            break

    pbar.close()
    print(f"[INFO] Totaal opgehaalde vergaderstukken: {len(all_docs)}")
    return all_docs


# Opslaan

def save_json(records):
    path = Path(OUTPUT_JSON)
    with path.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"[INFO] JSON opgeslagen in {path.resolve()}")

def save_csv(records):
    path = Path(OUTPUT_CSV)

    # kies een subset van velden die bijna altijd voorkomen
    fieldnames = [
        "id",
        "title",
        "introduction",
        "canonical",
        "dataurl",
        "frontenddate",
        "lastmodified",
        "available",
    ]

    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for d in records:
            row = {k: d.get(k) for k in fieldnames}
            w.writerow(row)

    print(f"[INFO] CSV opgeslagen in {path.resolve()}")



In [ ]:
docs = fetch_vergaderstukken()
save_json(docs)
save_csv(docs)

[INFO] Geen totaal beschikbaar — gebruik open-ended tqdm


Ophalen batches: 108it [06:47,  3.77s/it]

[INFO] Geen documenten meer (lege batch), klaar.
[INFO] Totaal opgehaalde vergaderstukken: 21558


[INFO] JSON opgeslagen in C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\scripts\policy_papers\beleidsnotas_rijksoverheid.json
[INFO] CSV opgeslagen in C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\scripts\policy_papers\beleidsnotas_rijksoverheid.csv


# cleaning df and optionally splitting

In [ ]:
df = pd.read_csv("beleidsnotas_rijksoverheid.csv")

#change frontenddate to clear date format
df['frontenddate'] = pd.to_datetime(df['frontenddate'], errors='coerce').dt.date

# show distribution of years in frontenddate
df['year'] = pd.DatetimeIndex(df['frontenddate']).year

# #delete frontenddate year column
df2 = df.drop(columns=["frontenddate"])

# delete zero after the year in 'year' column
df['year'] = df['year'].fillna(0).astype(int)

# counts = df['year'].value_counts().sort_index()
# print(counts)

#only keep columns with year 2015-2025
df_filtered = df[(df['year'] >= 2015) & (df['year'] <= 2025)]

# #how many rows dropped
# rows_dropped = len(df) - len(df_filtered)
# print(f"Rijen verwijderd: {rows_dropped}")

df_filtered = df_filtered.drop(columns={"frontenddate", "available", "introduction"})
df_filtered.info()

#---------------optional: split if files is too large to run

# #split dataframe into chunks of 500 rows
# chunk_size = 500
# num_chunks = (len(df_filtered) // chunk_size) + 1
# for i in range(num_chunks):
#     chunk = df_filtered[i*chunk_size:(i+1)*chunk_size]
#     chunk.to_csv(f"beleidsnotas_subset_part_{i+1}.csv", index=False)
#     print(f"[INFO] CSV opgeslagen in beleidsnotas_rijksoverheid_filtered_part_{i+1}.csv")

C:\Users\joly-\AppData\Local\Temp\ipykernel_24476\1677986887.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['frontenddate'] = pd.to_datetime(df['frontenddate'], errors='coerce').dt.date


<class 'pandas.core.frame.DataFrame'>
Index: 21552 entries, 6 to 21557
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            21552 non-null  object
 1   title         21552 non-null  object
 2   canonical     21552 non-null  object
 3   dataurl       21552 non-null  object
 4   lastmodified  21552 non-null  object
 5   year          21552 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 1.2+ MB
[INFO] CSV opgeslagen in beleidsnotas_rijksoverheid_filtered_part_1.csv
[INFO] CSV opgeslagen in beleidsnotas_rijksoverheid_filtered_part_2.csv
[INFO] CSV opgeslagen in beleidsnotas_rijksoverheid_filtered_part_3.csv
[INFO] CSV opgeslagen in beleidsnotas_rijksoverheid_filtered_part_4.csv
[INFO] CSV opgeslagen in beleidsnotas_rijksoverheid_filtered_part_5.csv
[INFO] CSV opgeslagen in beleidsnotas_rijksoverheid_filtered_part_6.csv
[INFO] CSV opgeslagen in beleidsnotas_rijksoverheid_filtered_part_7.csv
[INFO] 

## part 2: downloading pdfs from links in csv

In [3]:
#finding PDF links

def find_pdf_url(canonical_url: str) -> str | None:
    """
    Haalt de canonical pagina op en probeert een PDF-link te vinden.

    Werkt voor o.a.:
    - klassieke .pdf-links
    - open.overheid.nl/documenten/.../file (zoals jouw voorbeeld)
    - open.overheid.nl/documenten/.../pdf
    - links waarvan de tekst 'PDF' bevat
    """
    if not canonical_url:
        return None

    try:
        resp = requests.get(canonical_url, timeout=30)
        resp.raise_for_status()
    except Exception as e:
        print(f"[WARN] pagina niet bereikbaar: {canonical_url} ({e})")
        return None

    soup = BeautifulSoup(resp.text, "lxml")

    # 1) open.overheid.nl/documenten/.../file (meest betrouwbare voor veel docs)
    a = soup.select_one('a[href*="open.overheid.nl/documenten/"][href$="/file"]')
    if a and a.get("href"):
        return urljoin(canonical_url, a["href"])

    # 2) open.overheid.nl/documenten/.../pdf
    a = soup.select_one('a[href*="open.overheid.nl/documenten/"][href$="/pdf"]')
    if a and a.get("href"):
        return urljoin(canonical_url, a["href"])

    # 3) Klassieke .pdf-link ergens anders
    a = soup.select_one('a[href$=".pdf"], a[href*=".pdf"]')
    if a and a.get("href"):
        return urljoin(canonical_url, a["href"])

    # 4) Fallback: elk <a> met 'pdf' in de link-tekst
    for link in soup.find_all("a"):
        text = (link.get_text() or "").strip().lower()
        href = link.get("href")
        if "pdf" in text and href:
            return urljoin(canonical_url, href)

    print(f"[WARN] geen pdf-link gevonden op: {canonical_url}")
    return None


In [4]:
# 2) PDF downloaden en tekst extraheren
def extract_text_from_pdf_url(pdf_url: str) -> str | None:
    """
    Downloadt een PDF via pdf_url en geeft de samengevoegde tekst terug.
    Retourneert None als het niet lukt.
    """
    if not pdf_url:
        return None

    try:
        r = requests.get(pdf_url, timeout=60)
        r.raise_for_status()
    except Exception as e:
        print(f"[WARN] pdf niet te downloaden: {pdf_url} ({e})")
        return None

    try:
        with pdfplumber.open(BytesIO(r.content)) as pdf:
            texts = []
            for page in pdf.pages:
                texts.append(page.extract_text() or "")
        full_text = "\n".join(texts).strip()
        return full_text if full_text else None
    except Exception as e:
        print(f"[WARN] kon tekst niet extraheren uit pdf: {pdf_url} ({e})")
        return None

In [5]:

PDF_HREF_PATTERNS = [
    ".pdf",
    "/file",
    "/pdf",
    "/binaries/",
    "open.overheid.nl/documenten/",
]

def _pdf_links_in_soup(soup, base_url: str) -> list[str]:
    """
    Vind alle links in deze soup die 'pdf-achtig' zijn.
    """
    links = []
    for a in soup.find_all("a"):
        href = a.get("href")
        if not href:
            continue
        href_abs = urljoin(base_url, href)
        text = (a.get_text() or "").lower()

        looks_like_pdf = any(p in href_abs for p in PDF_HREF_PATTERNS) or "pdf" in text
        if looks_like_pdf:
            links.append(href_abs)

    # dedup met volgorde-behoud
    seen = set()
    out = []
    for u in links:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def find_pdf_urls(canonical_url: str) -> list[str]:
    """
    1) Zoek pdf-achtige links op de gegeven pagina.
    2) Zoek naar subdocument-links (/documenten/...) en volg die,
       om op die detailpagina's PDF-links te vinden.
    Retourneert ALLE gevonden PDF-URLs (uniek).
    """
    if not canonical_url:
        return []

    try:
        resp = requests.get(canonical_url, timeout=30)
        resp.raise_for_status()
    except Exception as e:
        print(f"[WARN] pagina niet bereikbaar: {canonical_url} ({e})")
        return []

    soup = BeautifulSoup(resp.text, "lxml")

    pdf_urls = []

    # --- Stap 1: direct PDF-achtige links op deze pagina ---
    pdf_urls.extend(_pdf_links_in_soup(soup, canonical_url))

    # --- Stap 2: subdocument-links volgen (/documenten/...) ---
    subpages = []
    for a in soup.select('a[href^="/documenten/"]'):
        href = a.get("href")
        if not href:
            continue
        sub_url = urljoin(canonical_url, href)
        subpages.append(sub_url)

    # dedup subpages, en voorkom dat we de canonical zelf nogmaals doen
    seen_sub = set()
    subpages_unique = []
    for u in subpages:
        if u not in seen_sub and u != canonical_url:
            seen_sub.add(u)
            subpages_unique.append(u)

    # nu elke subpagina ophalen en PDF-links zoeken
    for sub_url in subpages_unique:
        try:
            r2 = requests.get(sub_url, timeout=30)
            r2.raise_for_status()
        except Exception as e:
            print(f"[WARN] subpagina niet bereikbaar: {sub_url} ({e})")
            continue

        soup2 = BeautifulSoup(r2.text, "lxml")
        pdf_urls.extend(_pdf_links_in_soup(soup2, sub_url))

    # eind-deduplicatie
    seen = set()
    final = []
    for u in pdf_urls:
        if u not in seen:
            seen.add(u)
            final.append(u)

    if not final:
        print(f"[INFO] geen pdf-urls gevonden voor: {canonical_url}")

    return final


In [ ]:
#specify which part to process
# part = 44
# subset = pd.read_csv(f"beleidsnotas_subset_part_{part}.csv")
subset = pd.read_csv("doc")

pdf_texts = []

for canonical in tqdm(subset["canonical"], desc="PDF-tekst ophalen"):
    pdf_url = find_pdf_url(canonical)
    text = extract_text_from_pdf_url(pdf_url) if pdf_url else None
    pdf_texts.append(text)

subset["pdf_text"] = pdf_texts

PDF-tekst ophalen:  35%|███▍      | 18/52 [00:11<00:17,  1.99it/s]

[WARN] pagina niet bereikbaar: https://www.rijksoverheid.nl/documenten/beleidsnotas/2025/11/24/beslisnota-bij-kamervragen-over-het-artikel-cordaan-en-amsterdam-umc-stoppen-na-zeven-jaar-met-wijkkliniek-in-zuidoost%5B2%5D (400 Client Error: Bad Request for url: https://www.rijksoverheid.nl/documenten/beleidsnotas/2025/11/24/beslisnota-bij-kamervragen-over-het-artikel-cordaan-en-amsterdam-umc-stoppen-na-zeven-jaar-met-wijkkliniek-in-zuidoost%5B2%5D)


PDF-tekst ophalen: 100%|██████████| 52/52 [00:30<00:00,  1.69it/s]


In [7]:
# delete Nan rows in pdf_text column
subset = subset.dropna(subset=["pdf_text"])
#how many rows dropped
rows_dropped = len(pd.read_csv(f"beleidsnotas_subset_part_{part}.csv")) - len(subset)
print(f"Rijen verwijderd: {rows_dropped}")
#save to new csv
subset.to_csv(f"beleidsnotas_part_{part}_with_pdf_text.csv", index=False)

Rijen verwijderd: 14


In [10]:
df = pd.read_csv("beleidsnotas_rijksoverheid.csv")
df.head()

,id,title,introduction,canonical,dataurl,frontenddate,lastmodified,available
0,7847d28c-de67-44ec-8a9b-1e812a8fe046,Beslisnota bij Aanbiedingsbrief nota van wijzi...,<p>In een beslisnota staat achtergrondinformat...,https://www.rijksoverheid.nl/documenten/beleid...,https://opendata.rijksoverheid.nl/v1/documents...,0025-03-24T00:00:00.000Z,2025-04-07T15:00:12.468Z,2025-04-07T14:53:45.893Z
1,23e906e4-e6d3-4ea0-9813-4bb3e93ecd9f,Visie VNO-NCW op buisleidingen voor de industr...,<p>Advies van de werkgeversorganisatie VNO-NCW...,https://www.rijksoverheid.nl/documenten/beleid...,https://opendata.rijksoverheid.nl/v1/documents...,2009-07-01T15:28:00.000Z,2025-09-10T12:41:48.788Z,2009-07-01T15:27:00.000Z
2,1f16b183-dd64-41b0-bc07-7515b56bd47c,Algemeen beleidskader indeplaatsstelling bij t...,<p>Dit beleidskader bevat de uitgangspunten vo...,https://www.rijksoverheid.nl/documenten/beleid...,https://opendata.rijksoverheid.nl/v1/documents...,2011-03-07T23:00:00.000Z,2025-09-10T12:42:33.047Z,2011-03-08T11:55:00.000Z
3,cf3a0e47-db4a-4d1a-9608-3e98509d76f7,Slotbeschouwing uit de essaybundel 'Veiligheid...,<p>&#39;Samenvattende analyse: belemmerende ov...,https://www.rijksoverheid.nl/documenten/beleid...,https://opendata.rijksoverheid.nl/v1/documents...,2011-04-13T13:21:00.000Z,2025-09-10T12:43:14.002Z,2011-04-13T13:21:00.000Z
4,26252414-82f7-41e1-9aab-74b4825116e3,Handhavingbeleidsplannen bouwregelgeving,<p>Gemeenten handhaven de bouwregelgeving en l...,https://www.rijksoverheid.nl/documenten/beleid...,https://opendata.rijksoverheid.nl/v1/documents...,2013-11-12T10:10:00.000Z,2025-09-10T12:44:03.581Z,2013-11-12T10:08:00.000Z
